# Source Data Profiling

Data quality checks on incoming shipment orders, tracking events and delivery attempts CSVs before processing. Each check records its result; section 5 collects them into one summary table.

## 1. Setup

In [0]:
%sql
use catalog logistics;
use schema bronze;

In [0]:
shipment_orders_path="/Volumes/logistics/bronze/volume/transaction_data/shipment_orders/"
# events_data folder holds both tracking events and delivery attempts CSVs
tracking_events="/Volumes/logistics/bronze/volume/events_data/shipment_tracking_events*.csv"
delivery_attempts="/Volumes/logistics/bronze/volume/events_data/delivery_attempts*.csv"

In [0]:
from pyspark.sql import functions as F
from pyspark.sql.window import Window

# ---------------- Configuration ----------------
TS_FMT                = "yyyy-MM-dd'T'HH:mm:ss'Z'"
LATE_THRESHOLD_MIN    = 120   # received_ts - event_ts above this => late-arriving
IQR_MULTIPLIER        = 3     # outlier fence = Q1/Q3 -/+ k * IQR
VALID_SHIPMENT_STATUS = ["CREATED", "IN_TRANSIT", "OUT_FOR_DELIVERY", "DELIVERED"]
VALID_ATTEMPT_RESULTS = ["DELIVERED", "FAILED", "RESCHEDULED"]

# ---------------- Results collector ----------------
dq_results = {}  # (source, check) -> value; re-running a cell overwrites its entry

def record(source, check, value):
    dq_results[(source, check)] = None if value is None else float(value)
    print(f"[{source}] {check}: {value}")

# ---------------- Column helpers ----------------
def parse_ts(c):
    return F.try_to_timestamp(F.col(c), F.lit(TS_FMT))

def is_blank(c):
    return F.col(c).isNull() | (F.trim(F.col(c)) == "")

def invalid_ts(c):
    """Missing, unparseable, or in the future."""
    return is_blank(c) | parse_ts(c).isNull() | (parse_ts(c) > F.current_timestamp())

def count_if(cond, alias):
    return F.sum(F.when(cond, 1).otherwise(0)).alias(alias)

# ---------------- DataFrame helpers ----------------
def read_csv(path):
    """Read all CSVs under a path as strings (no silent type coercion) + source file name."""
    return (spark.read.option("header", True).option("inferSchema", False).csv(path)
            .withColumn("_source_file", F.col("_metadata.file_name")))

def inferred_types(path):
    return spark.read.option("header", True).option("inferSchema", True).csv(path).dtypes

def missing_profile(df, cols):
    total = df.count()
    row = df.agg(*[count_if(is_blank(c), c) for c in cols]).first().asDict()
    rows = [(c, int(row[c]), round(100.0 * row[c] / total, 2)) for c in cols]
    return (spark.createDataFrame(rows, "column string, missing_count long, missing_pct double")
            .orderBy(F.desc("missing_count")))

def duplicate_row_count(df, cols):
    return df.count() - df.select(*cols).distinct().count()

def duplicate_keys(df, keys):
    return df.groupBy(*keys).count().filter("count > 1")

In [0]:
# Raw (all-string) reads
so_raw = read_csv(shipment_orders_path)
te_raw = read_csv(tracking_events)
da_raw = read_csv(delivery_attempts)

# Business columns (exclude the lineage column) - computed once
SO_COLS = [c for c in so_raw.columns if c != "_source_file"]
TE_COLS = [c for c in te_raw.columns if c != "_source_file"]
DA_COLS = [c for c in da_raw.columns if c != "_source_file"]

# Typed working views
so = so_raw.withColumns({
    "created_ts_p":  parse_ts("created_ts"),
    "promised_ts_p": parse_ts("promised_delivery_ts"),
    "weight_kg":     F.col("package_weight_kg").try_cast("double"),
    "cost":          F.col("shipping_cost").try_cast("double"),
})
te = te_raw.withColumns({
    "event_ts_p":    parse_ts("event_ts"),
    "received_ts_p": parse_ts("received_ts"),
})
da = da_raw.withColumns({
    "attempt_ts_p":    parse_ts("attempt_ts"),
    "received_ts_p":   parse_ts("received_ts"),
    "attempt_seq_num": F.regexp_extract("attempt_sequence", r"(\d+)$", 1).try_cast("int"),  # 'ATTEMP_4' -> 4
})
shipment_keys = so_raw.select("shipment_id").distinct()

# File inventory
file_inventory = (so_raw.groupBy("_source_file").count().withColumn("source", F.lit("shipment_orders"))
    .unionByName(te_raw.groupBy("_source_file").count().withColumn("source", F.lit("tracking_events")))
    .unionByName(da_raw.groupBy("_source_file").count().withColumn("source", F.lit("delivery_attempts")))
    .select("source", "_source_file", F.col("count").alias("records"))
    .orderBy("source", "_source_file"))
display(file_inventory)

## 2. Shipment orders

In [0]:
SRC = "shipment_orders"
r = so.agg(F.count("*").alias("total"),
           F.countDistinct("shipment_id").alias("distinct_ids"),
           count_if(is_blank("shipment_id"), "missing_ids")).first()

record(SRC, "total_records", r.total)
record(SRC, "distinct_shipment_ids", r.distinct_ids)
record(SRC, "missing_shipment_ids", r.missing_ids)

In [0]:
SRC = "shipment_orders"
so_dup_keys = duplicate_keys(so, ["shipment_id"])

record(SRC, "duplicate_rows", duplicate_row_count(so_raw, SO_COLS))
record(SRC, "duplicate_shipment_ids", so_dup_keys.count())

# Every copy of a duplicated shipment_id, with the file it came from
display(so.join(so_dup_keys.select("shipment_id"), "shipment_id")
          .select(*SO_COLS, "_source_file")
          .orderBy("shipment_id", "_source_file"))

In [0]:
SRC = "shipment_orders"
so_missing = missing_profile(so_raw, SO_COLS)
record(SRC, "rows_with_any_missing", so_raw.filter(F.greatest(*[is_blank(c).cast("int") for c in SO_COLS]) == 1).count())
display(so_missing)

In [0]:
SRC = "shipment_orders"

# Spark-inferred type per column
display(spark.createDataFrame(inferred_types(shipment_orders_path), "column string, inferred_type string"))

# Values that do not conform to the expected type
r = so.agg(
    count_if(~is_blank("package_weight_kg") & F.col("weight_kg").isNull(), "bad_weight"),
    count_if(~is_blank("shipping_cost") & F.col("cost").isNull(), "bad_cost"),
    count_if(invalid_ts("created_ts"), "bad_created"),
    count_if(invalid_ts("promised_delivery_ts"), "bad_promised"),
    count_if(F.col("promised_ts_p") < F.col("created_ts_p"), "promised_before_created"),
).first()

record(SRC, "non_numeric_package_weight", r.bad_weight)
record(SRC, "non_numeric_shipping_cost", r.bad_cost)
record(SRC, "invalid_created_ts", r.bad_created)
record(SRC, "invalid_promised_delivery_ts", r.bad_promised)
record(SRC, "promised_before_created", r.promised_before_created)

In [0]:
SRC = "shipment_orders"
is_valid_status = F.upper(F.trim("shipment_status")).isin(VALID_SHIPMENT_STATUS)

display(so.groupBy("shipment_status")
          .agg(F.count("*").alias("records"))
          .withColumn("is_valid", F.coalesce(F.upper(F.trim("shipment_status")).isin(VALID_SHIPMENT_STATUS), F.lit(False)))
          .orderBy(F.desc("records")))

so_invalid_status = so.filter(is_blank("shipment_status") | ~is_valid_status)
record(SRC, "invalid_shipment_status", so_invalid_status.count())
display(so_invalid_status.select(*SO_COLS, "_source_file"))

In [0]:
SRC = "shipment_orders"

# Distribution
display(so.select(
    *[F.percentile_approx(m, p).alias(f"{m}_p{int(p*100):02d}") for m in ["weight_kg", "cost"] for p in [0.0, 0.01, 0.25, 0.5, 0.75, 0.99, 1.0]]))

# IQR fences
q = so.select(F.percentile_approx("weight_kg", [0.25, 0.75]).alias("w"),
              F.percentile_approx("cost", [0.25, 0.75]).alias("c")).first()
fence = lambda q1, q3: (q1 - IQR_MULTIPLIER * (q3 - q1), q3 + IQR_MULTIPLIER * (q3 - q1))
w_lo, w_hi = fence(*q.w)
c_lo, c_hi = fence(*q.c)
print(f"weight fence: [{w_lo:.2f}, {w_hi:.2f}]   cost fence: [{c_lo:.2f}, {c_hi:.2f}]")

so_flagged = so.withColumns({
    "extreme_weight": (F.col("weight_kg") <= 0) | ~F.col("weight_kg").between(w_lo, w_hi),
    "extreme_cost":   (F.col("cost") <= 0)      | ~F.col("cost").between(c_lo, c_hi),
})
r = so_flagged.agg(count_if(F.col("extreme_weight"), "w"), count_if(F.col("extreme_cost"), "c")).first()
record(SRC, "extreme_package_weight", r.w)
record(SRC, "extreme_shipping_cost", r.c)

display(so_flagged.filter("extreme_weight OR extreme_cost")
                  .select("shipment_id", "package_weight_kg", "shipping_cost", "extreme_weight", "extreme_cost", "_source_file")
                  .orderBy(F.desc("weight_kg"), F.desc("cost")))

## 3. Tracking events

In [0]:
SRC = "tracking_events"
r = te.agg(F.count("*").alias("total"),
           F.countDistinct("tracking_event_id").alias("distinct_ids"),
           count_if(is_blank("tracking_event_id"), "missing_ids")).first()

record(SRC, "total_records", r.total)
record(SRC, "distinct_tracking_event_ids", r.distinct_ids)
record(SRC, "missing_tracking_event_ids", r.missing_ids)
display(missing_profile(te_raw, TE_COLS))

In [0]:
SRC = "tracking_events"
te_orphans = te.filter(~is_blank("shipment_id")).join(shipment_keys, "shipment_id", "left_anti")

record(SRC, "missing_shipment_ids", te.filter(is_blank("shipment_id")).count())
record(SRC, "shipment_ids_not_in_orders", te_orphans.count())
display(te_orphans.select(*TE_COLS, "_source_file"))

In [0]:
SRC = "tracking_events"
r = te.agg(
    count_if(is_blank("event_ts"), "missing"),
    count_if(~is_blank("event_ts") & F.col("event_ts_p").isNull(), "unparseable"),
    count_if(F.col("event_ts_p") > F.current_timestamp(), "future"),
    count_if(invalid_ts("received_ts"), "bad_received"),
    count_if(F.col("received_ts_p") < F.col("event_ts_p"), "received_before_event"),
).first()

record(SRC, "missing_event_ts", r.missing)
record(SRC, "unparseable_event_ts", r.unparseable)
record(SRC, "future_event_ts", r.future)
record(SRC, "invalid_received_ts", r.bad_received)
record(SRC, "received_before_event", r.received_before_event)

display(te.filter(invalid_ts("event_ts") | invalid_ts("received_ts") | (F.col("received_ts_p") < F.col("event_ts_p")))
          .select(*TE_COLS, "_source_file"))

In [0]:
SRC = "tracking_events"
record(SRC, "missing_event_type", te.filter(is_blank("event_type")).count())
display(te.groupBy("event_type").agg(F.count("*").alias("records"),
                                     F.countDistinct("shipment_id").alias("shipments"))
          .orderBy(F.desc("records")))

In [0]:
SRC = "tracking_events"
te_dup_ids = duplicate_keys(te, ["tracking_event_id"])
te_dup_biz = duplicate_keys(te, ["shipment_id", "event_type", "event_ts"])  # same event under different IDs

record(SRC, "duplicate_rows", duplicate_row_count(te_raw, TE_COLS))
record(SRC, "duplicate_tracking_event_ids", te_dup_ids.count())
record(SRC, "duplicate_business_events", te_dup_biz.count())

display(te.join(te_dup_ids.select("tracking_event_id"), "tracking_event_id")
          .select(*TE_COLS, "_source_file").orderBy("tracking_event_id"))

In [0]:
SRC = "tracking_events"
w_arrival = (Window.partitionBy("shipment_id").orderBy("received_ts_p")
                   .rowsBetween(Window.unboundedPreceding, -1))

te_late = te.withColumns({
    "lag_minutes": (F.unix_timestamp("received_ts_p") - F.unix_timestamp("event_ts_p")) / 60,
    "max_prior_event_ts": F.max("event_ts_p").over(w_arrival),
}).withColumns({
    "late_by_threshold": F.col("lag_minutes") > LATE_THRESHOLD_MIN,
    "out_of_order": F.col("event_ts_p") < F.col("max_prior_event_ts"),
})

# Ingestion lag distribution
display(te_late.select(*[F.percentile_approx("lag_minutes", p).alias(f"lag_p{int(p*100):02d}") for p in [0.0, 0.5, 0.95, 0.99, 1.0]]))

r = te_late.agg(count_if(F.col("late_by_threshold"), "late"), count_if(F.col("out_of_order"), "ooo")).first()
record(SRC, f"late_events_gt_{LATE_THRESHOLD_MIN}_min", r.late)
record(SRC, "out_of_order_events", r.ooo)

display(te_late.filter("late_by_threshold OR out_of_order")
               .select("tracking_event_id", "shipment_id", "event_type", "event_ts", "received_ts",
                       "lag_minutes", "late_by_threshold", "out_of_order", "_source_file")
               .orderBy(F.desc("lag_minutes")))

## 4. Delivery attempts

In [0]:
SRC = "delivery_attempts"
r = da.agg(F.count("*").alias("total"),
           F.countDistinct("attempt_id").alias("distinct_ids"),
           count_if(is_blank("attempt_id"), "missing_ids")).first()

record(SRC, "total_records", r.total)
record(SRC, "distinct_attempt_ids", r.distinct_ids)
record(SRC, "missing_attempt_ids", r.missing_ids)
record(SRC, "duplicate_rows", duplicate_row_count(da_raw, DA_COLS))
record(SRC, "duplicate_attempt_ids", duplicate_keys(da, ["attempt_id"]).count())
display(missing_profile(da_raw, DA_COLS))

In [0]:
SRC = "delivery_attempts"
da_orphans = da.filter(~is_blank("shipment_id")).join(shipment_keys, "shipment_id", "left_anti")

record(SRC, "missing_shipment_ids", da.filter(is_blank("shipment_id")).count())
record(SRC, "shipment_ids_not_in_orders", da_orphans.count())
display(da_orphans.select(*DA_COLS, "_source_file"))

In [0]:
SRC = "delivery_attempts"
is_valid_result = F.coalesce(F.upper(F.trim("attempt_result")).isin(VALID_ATTEMPT_RESULTS), F.lit(False))

record(SRC, "invalid_attempt_result", da.filter(~is_valid_result).count())
record(SRC, "delivered_without_pod", da.filter((F.col("attempt_result") == "DELIVERED") & (F.col("proof_of_delivery") != "Y")).count())

display(da.groupBy("attempt_result").agg(F.count("*").alias("records"))
          .withColumn("is_valid", F.coalesce(F.upper(F.trim("attempt_result")).isin(VALID_ATTEMPT_RESULTS), F.lit(False)))
          .orderBy(F.desc("records")))

In [0]:
SRC = "delivery_attempts"
r = da.agg(
    count_if((F.col("attempt_result") == "DELIVERED") & ~is_blank("failure_reason"), "delivered_with_reason"),
    count_if((F.col("attempt_result") == "FAILED") & is_blank("failure_reason"), "failed_without_reason"),
).first()

record(SRC, "delivered_with_failure_reason", r.delivered_with_reason)
record(SRC, "failed_without_failure_reason", r.failed_without_reason)

# Failure reason by attempt result
display(da.groupBy("failure_reason", "attempt_result").agg(F.count("*").alias("records"))
          .orderBy("failure_reason", "attempt_result"))

In [0]:
SRC = "delivery_attempts"

# Per-shipment sequence integrity: must be 1..n with no gaps or repeats
seq_by_shipment = (da.groupBy("shipment_id")
                     .agg(F.count("*").alias("attempts"),
                          F.countDistinct("attempt_seq_num").alias("distinct_seq"),
                          F.min("attempt_seq_num").alias("min_seq"),
                          F.max("attempt_seq_num").alias("max_seq"))
                     .filter("min_seq <> 1 OR max_seq <> attempts OR distinct_seq <> attempts"))

# Sequence must increase with attempt time
w_time = Window.partitionBy("shipment_id").orderBy("attempt_ts_p", "attempt_seq_num")
seq_order_issues = (da.withColumn("prev_seq", F.lag("attempt_seq_num").over(w_time))
                      .filter(F.col("attempt_seq_num") <= F.col("prev_seq")))

record(SRC, "unparseable_attempt_sequence", da.filter(F.col("attempt_seq_num").isNull()).count())
record(SRC, "shipments_with_sequence_gaps_or_repeats", seq_by_shipment.count())
record(SRC, "sequence_out_of_time_order", seq_order_issues.count())

display(da.groupBy("attempt_sequence", "attempt_seq_num").agg(F.count("*").alias("records")).orderBy("attempt_seq_num"))
display(seq_by_shipment.orderBy("shipment_id"))

In [0]:
SRC = "delivery_attempts"
da_created = da.join(so.select("shipment_id", "created_ts_p").dropDuplicates(["shipment_id"]), "shipment_id", "left")

r = da_created.agg(
    count_if(is_blank("attempt_ts"), "missing"),
    count_if(~is_blank("attempt_ts") & F.col("attempt_ts_p").isNull(), "unparseable"),
    count_if(F.col("attempt_ts_p") > F.current_timestamp(), "future"),
    count_if(invalid_ts("received_ts"), "bad_received"),
    count_if(F.col("received_ts_p") < F.col("attempt_ts_p"), "received_before_attempt"),
    count_if(F.col("attempt_ts_p") < F.col("created_ts_p"), "before_created"),
).first()

record(SRC, "missing_attempt_ts", r.missing)
record(SRC, "unparseable_attempt_ts", r.unparseable)
record(SRC, "future_attempt_ts", r.future)
record(SRC, "invalid_received_ts", r.bad_received)
record(SRC, "received_before_attempt", r.received_before_attempt)
record(SRC, "attempt_before_shipment_created", r.before_created)

display(da.filter(invalid_ts("attempt_ts") | invalid_ts("received_ts") | (F.col("received_ts_p") < F.col("attempt_ts_p")))
          .select(*DA_COLS, "_source_file"))

## 5. Data quality summary

In [0]:
dq_summary = spark.createDataFrame(
    [(src, chk, val) for (src, chk), val in dq_results.items()],
    "source string, check string, value double")

display(dq_summary.orderBy("source", "check"))